country isn't updated anymore so we are using an old year 

In [1]:
platformID = 'TWI'

In [2]:
from datetime import datetime
import pandas as pd

import psycopg2


## import helper

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info

from functions import execute_sql_query
import test_functions

In [4]:
# country
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID', 
                              keep_default_na=False)

# week 
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])
week_tester['week_ending'] = pd.to_datetime(week_tester['week_ending'])

# social media accounts
dtype_dict = {'Channel ID': 'str',
              'Linked FB Account': 'str'}
socialmedia_accounts = pd.read_excel(f"../../{gam_info['lookup_file']}", dtype=dtype_dict,
                                     sheet_name='Social Media Accounts new')

socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Year'] == gam_info['file_timeinfo']]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['PlatformID'] == {platformID}]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Status'] == 'active']
socialmedia_accounts['Channel ID'] = socialmedia_accounts['Channel ID'].dropna().apply(lambda x: str(int(x)))

channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
formatted_channel_ids = ', '.join(f"'{channel_id}'" for channel_id in channel_ids)


# ingestion

# country

In [5]:
# takes forever to load
tw_country_df = pd.read_excel(f'../data/raw/{platformID}/stale_Twitter Engagements inc country.xlsx')

# ensure the accounts are strings (TW & FB)
tw_country_df['TW Account ID'] = tw_country_df['TW Account ID'].astype(str)
tw_country_df['TW Linked FB account'] = tw_country_df['TW Linked FB account'].astype(str).str.split('.').str[0]

tw_country_df = tw_country_df.rename(columns={"TW Account ID": "tw_account_id"})
tw_country_df = tw_country_df.rename(columns={'Week Number': 'WeekNumber_finYear'})

tw_country_df.sample()

,tw_account_id,TW Account Name,TW Account Handle,Weekly Engagements,Weekly Video Views,WeekNumber_finYear,Actual Week,TW Linked FB account,TW Studios Exc UK,Country,Engagement %,Week Commencing,TW Service Code
107761,2613311137,Niambie,NiambieTZ,44,0,35,48,519260718174772,NaN,Poland,0.005,2023-11-27,MA-


In [6]:
# test accounts 
column_name = 'tw_account_id'
test_functions.test_filter_elements_returned(tw_country_df, channel_ids, column_name, 
                                             '1_TW_9', test_step= 'stale country dataset - channels')

# 

...testing tw_account_id...
Test 1_TW_9 passed: everything found!
...updating logbook...



In [7]:
tw_country_df.to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country.csv", 
                     index=None, na_rep='')